In [ ]:
library(tidyr)
library(ggplot2)
library(ggh4x)
library(dplyr)
library(RColorBrewer)
library(tidyselect)
library(tidyr)
library(stringr)
library(ggpubr)
library(ggdist)
library(labdsv)
library(vegan)
library(readr)
library(ape)
library(lme4)
library(lsmeans)
library(scales)
library(purrr)
library(broom)
library(Polychrome)
library(cowplot)
library(reshape2)
library(doParallel)

In [ ]:
metadata <- read.csv("DOME-Final-16S-Nasal-Metadata.csv")

metadata <- metadata %>%
  dplyr::mutate(Sample.ID = stringr::str_remove(Sample.ID, "^19-"))

metadata<- metadata %>%
  # Cohort based on Sample.ID prefix
  mutate(Cohort = case_when(
    startsWith(Sample.ID, "C") ~ "Cow",
    startsWith(Sample.ID, "D") ~ "Non-Farmer",
    startsWith(Sample.ID, "W") ~ "Farmer",
    TRUE ~ "Unknown"
  ))

In [ ]:
GENUS <- read.csv("DOME-16S-Taxa-Counts-Genus.csv"))

In [ ]:
relative_abundance_genus <- asv_counts %>%
  tibble::column_to_rownames("Sample.ID") %>%
  { 
    rs <- rowSums(.)
    rs[rs == 0] <- 1  
    sweep(., 1, rs, "/")
  } %>%
  as.data.frame() %>%
  tibble::rownames_to_column("Sample.ID")

In [ ]:
# Apply 0.05% cutoff
relative_abundance_genus <- relative_abundance_genus %>%
  dplyr::mutate(across(-Sample.ID, ~ ifelse(.x <= 0.0005, 0, .x)))

# Merge with metadata
merged_GENUS <- right_join(metadata, relative_abundance_genus, by = "Sample.ID")

In [ ]:
# Drop metadata columns (everything up to Sample.Type, as before)
abund <- merged_GENUS %>%
  select(-c(Sample.ID, Subject, Site, Season, Month, Year, Collection.Date, Sample.Type))

# Put sample IDs as rownames
rownames(abund) <- merged_GENUS$Sample.ID

In [ ]:
richness <- diversity(abund, index = "simpson")
richness <- data.frame(Sample.ID = rownames(abund), Simpson.Index = richness)
richness_meta <- richness %>%
  left_join(merged_GENUS %>% select(Sample.ID, Subject, Season, Month, Year, Sample.Type),
            by = "Sample.ID")

In [ ]:
richness_meta <- richness_meta %>%
  mutate(Cohort = case_when(
    startsWith(Sample.ID, "C") ~ "Cow",
    startsWith(Sample.ID, "D") ~ "Non-Farmer",
    startsWith(Sample.ID, "W") ~ "Farmer",
    TRUE ~ "Unknown"
    ),
    Season = factor(
      Season,
      levels = c("Spring", "Summer", "Autumn", "Winter")  
    )
)

In [ ]:
richness_over_time <- ggplot(richness_meta, aes(x = Season, y = Simpson.Index, fill = Cohort)) +
  # Boxplot
  geom_boxplot(
    outlier.shape = NA,
    alpha = 0.8,
    color = "black",
    width = 0.5,
    position = position_dodge(width = 0.8)
  ) +
guides(fill = guide_legend(override.aes = list(color = NA, alpha = 1))) +
  # Half-eye (raincloud)
  stat_halfeye(
    aes(fill = Cohort),
    alpha = 0.4,
    width = 0.6,
    justification = -0.25,
    point_colour = NA,
    position = position_dodge(width = 0.8)
  ) +
  # Jittered points
  geom_jitter(
    aes(color = Cohort),  
    size = .25,
    alpha = 0.75,
    position = position_jitterdodge(dodge.width = 0.8, jitter.width = 0.15)
  ) +
  scale_fill_manual(values = defined_colors) +
  scale_color_manual(values = defined_colors) +  
  facet_wrap(~ Year, ncol = 1, scales = "free_y") +
  theme_bw() +
  labs(y = "Simpson Index", x = "Season") +
  theme(
    panel.grid = element_blank(),               
    strip.background = element_rect(fill = "black"),
    strip.text = element_text(color = "white", face = "bold", size = 12),
    axis.text.x = element_text(size = 8),
    axis.title.x = element_blank(),
    text = element_text(family = "Helvetica", size = 12)
  ) 

In [ ]:
ggplot2::ggsave("16S-Simpson-Index-Longitudinal.pdf", richness_over_time, width = 9, height = 12, dpi = 360, units = "in")